# Sumativa 3 · Punto 1 · Preparación de datos y manejo de faltantes

**MCDI501 · Estadística Computacional para la Toma de Decisiones · Grupo 6**

Responsable de esta sección: **Fernanda Ovalle Román**

Docente: Jean Paul Maidana González · Universidad Andrés Bello · UNAB Online

---

## Qué cubre este cuaderno

Es el criterio 1 de la pauta (21 pts). Cuatro tareas:

1. **Resumir el diagnóstico de la S1** sobre `anio_ing_carr_ori` y
   `sem_ing_carr_ori`, sin recalcularlo, y nombrar el mecanismo de pérdida.
2. **Imputar por regresión lineal múltiple**, verificando los supuestos
   (normalidad de residuos, homocedasticidad, colinealidad) y reportando
   R² y RMSE.
3. **Comparar las tres estrategias** —eliminación, imputación simple e
   imputación por regresión— en tamaño muestral, momentos de la distribución
   y preservación de las correlaciones validadas en S1 y S2.
4. **Justificar qué conjunto se usa** como base del modelamiento logístico,
   y dejar la partición 70/30 lista para el punto 2.

Todo el cálculo vive en `S3/src/preparacion.py`. El cuaderno orquesta y
documenta; no define lógica.

## Requisitos

```
pip install statsmodels scikit-learn
```

El CSV crudo (175 MB) no se versiona. El módulo lo busca en las rutas
habituales del proyecto; si no lo encuentra, pásalo explícitamente a
`cargar_base(ruta=...)`.

## 1 · Entorno y carga

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "S3").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "S3" / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import preparacion as P

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("raíz:", RAIZ)

In [ ]:
df = P.cargar_base()

print(f"Filas tras el filtro de pregrado universitario: {len(df):,}".replace(",", "."))
print(f"Debe coincidir con la S1:                       105.063")
df[["semestres_reales", "semestres_desde_act", "dur_total_carr",
    "edad_titulacion", "atraso_semestres"]].describe().round(2)

## 2 · Diagnóstico de faltantes

La S1 identificó que `anio_ing_carr_ori` y `sem_ing_carr_ori` traen los
centinelas `1900` y `0` en el 12,75 % de los casos. La pauta pide resumir ese
diagnóstico, no repetirlo, de modo que aquí solo se reproduce el conteo para
verificar la trazabilidad y se agrega lo que la S1 dejó pendiente: **nombrar
el mecanismo de pérdida**.

In [ ]:
dx = P.diagnostico_faltantes(df)

print(f"Casos sin año de ingreso original : {dx['n_faltantes']:,}".replace(",", "."))
print(f"Casos con atraso calculable       : {dx['n_completos']:,}".replace(",", "."))
print(f"La S1 reportó                     : 91.667")
dx["resumen"]

### El mecanismo no es aleatorio

La distinción importa porque decide qué está permitido hacer. Si la ausencia
fuera completamente al azar (MCAR), eliminar los casos incompletos no
introduciría sesgo y el análisis de casos completos sería válido. Si depende
de variables observadas (MAR), eliminarlos **sí** sesga, y la imputación
condicionada a esas variables es el procedimiento correcto.

La comprobación es directa: ¿la proporción de faltantes depende de alguna
variable observada?

In [ ]:
print(dx["por_jornada"].to_string(index=False))
print()
print(f"Razón entre la jornada con más y con menos faltantes: {dx['razon_max_min']}×")
print(f"Mecanismo: {dx['mecanismo']}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
t = dx["por_jornada"].sort_values("pct_sin_info")
colores = ["#c0c0c0" if j != "Diurna" else "#d95f02" for j in t["jornada"]]
ax.barh(t["jornada"], t["pct_sin_info"], color=colores)
for y, (v, n) in enumerate(zip(t["pct_sin_info"], t["n"])):
    ax.text(v + 1, y, f"{v} %  (n = {n:,})".replace(",", "."), va="center", fontsize=9)
ax.set_title("El dato faltante no está repartido al azar: la jornada diurna\n"
             "casi no lo tiene y la modalidad a distancia lo tiene en tres de cada cuatro",
             fontsize=12, pad=14)
ax.set_xlabel("Registros sin año de ingreso original (%)")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(RAIZ / "S3/informe/figuras/s3_01_faltantes_por_jornada.png", dpi=150)
plt.show()

**Lectura.** La proporción de faltantes va del 1,5 % en la jornada diurna al
75,9 % en la modalidad a distancia: una razón de **50 veces**. La ausencia
depende de una variable observada, de modo que el mecanismo es **MAR** y no
MCAR.

Dos consecuencias prácticas, y son las que justifican todo lo que sigue:

1. **Eliminar los casos incompletos sesga la muestra.** No elimina un 12,75 %
   repartido parejo: elimina tres cuartos de la modalidad a distancia y casi
   nada de la diurna. El conjunto resultante sobrerrepresenta a la jornada
   diurna.
2. **La imputación condicional es legítima**, precisamente porque la ausencia
   depende de variables que sí observamos.

## 3 · Imputación por regresión lineal múltiple

### Qué se imputa, y con qué

Lo que falta es el año y el semestre de ingreso **original**. De ellos depende
`semestres_reales`, y de esa depende `atraso_semestres`, que es la variable
sobre la que se construye el objetivo del modelo logístico. Se modela
directamente `semestres_reales`, que es la cantidad que interesa y la única
cuyo ajuste admite interpretación en semestres.

Los predictores son cuatro, dentro del rango de 3 a 5 que fija la pauta,
elegidos por dos criterios: estar observados en el 100 % de los casos
incompletos, y tener respaldo en las correlaciones de S1 validadas en S2.

| Predictor | Por qué |
|---|---|
| `semestres_desde_act` | Duración calculada desde el ingreso a la carrera **actual**, observado en todos los casos. Es el sustituto natural del dato ausente. |
| `dur_total_carr` | Duración formal del plan. ρ = 0,46 con los semestres reales (S2). |
| `edad_titulacion` | ρ = 0,47 con el atraso, la correlación más fuerte validada en S2. |
| `area_conocimiento` | La S2 mostró heterogeneidad por área en la brecha de atraso. |

El hallazgo que hace viable la imputación es que **`anio_ing_carr_act` está
observado en el 100 % de los 13.396 casos incompletos**, y coincide con el
original en el 91,2 % de los casos completos. El dato que falta es, en buena
medida, recuperable desde uno que sí está.

In [ ]:
ajuste = P.ajustar_imputador(df)

print(f"Observaciones de ajuste : {ajuste['n_ajuste']:,}".replace(",", "."))
print(f"R²                      : {ajuste['r2']:.4f}")
print(f"R² ajustado             : {ajuste['r2_aj']:.4f}")
print(f"RMSE                    : {ajuste['rmse']:.3f} semestres")
print(f"MAE                     : {ajuste['mae']:.3f} semestres")

In [ ]:
ajuste["coeficientes"].head(8)

### Verificación de supuestos

In [ ]:
print(ajuste["supuestos"].to_string(index=False))
print()
print("Factor de inflación de la varianza, por predictor:")
print(ajuste["supuestos"].attrs["vif"].to_string(index=False))

In [ ]:
import scipy.stats as st

modelo = ajuste["modelo"]
resid = modelo.resid
pred = modelo.fittedvalues

fig, ax = plt.subplots(1, 3, figsize=(14, 4))

ax[0].hist(resid[np.abs(resid) < 15], bins=60, color="#c0c0c0", edgecolor="white")
ax[0].set_title("Residuos: cola larga a la derecha")
ax[0].set_xlabel("Residuo (semestres)")

st.probplot(resid.sample(5000, random_state=42), dist="norm", plot=ax[1])
ax[1].set_title("Q-Q: se aparta en los extremos")
ax[1].get_lines()[0].set_color("#999999")
ax[1].get_lines()[1].set_color("#d95f02")

m = np.abs(resid) < 15
ax[2].scatter(pred[m][::40], resid[m][::40], s=3, alpha=.2, color="#777777")
ax[2].axhline(0, color="#d95f02", lw=1)
ax[2].set_title("Varianza no constante")
ax[2].set_xlabel("Valor ajustado"); ax[2].set_ylabel("Residuo")

plt.tight_layout()
plt.savefig(RAIZ / "S3/informe/figuras/s3_02_supuestos.png", dpi=150)
plt.show()

**Lectura de los supuestos, sin maquillaje.**

**Colinealidad: se cumple.** El VIF máximo es 2,42 y todos los predictores
quedan por debajo de 2,5, lejos del umbral habitual de 5. Las cuatro variables
aportan información distinta.

**Normalidad de los residuos: no se cumple.** La asimetría es 7,6 y la
curtosis 106. La causa es sustantiva, no técnica: existe un grupo pequeño de
titulaciones con duraciones muy superiores a lo que los predictores anticipan
—reingresos, carreras interrumpidas y retomadas— que el modelo lineal no puede
reproducir y deja como residuos grandes y positivos.

**Homocedasticidad: no se cumple.** Breusch-Pagan rechaza con holgura; la
dispersión de los residuos crece con el valor ajustado.

**Qué significa esto para el uso que le damos.** Las dos violaciones afectan a
la *inferencia*: con residuos no normales y varianza no constante, los errores
estándar y los valores p de la tabla de coeficientes dejan de ser confiables.
Pero aquí el modelo no se usa para inferir sobre los coeficientes, sino para
**predecir** un valor ausente, y la estimación por mínimos cuadrados sigue
siendo insesgada y consistente para la media condicional aunque esos supuestos
fallen. Lo que importa para imputar es el error de predicción, y el MAE de
**0,98 semestres** dice que la imputación típica se equivoca en menos de un
semestre.

La cautela que sí corresponde: el modelo subestima sistemáticamente las
duraciones extremas, de modo que la imputación **comprime la cola derecha**.
La sección 4 mide ese efecto en lugar de suponerlo.

## 4 · Las tres estrategias

| | Estrategia | Qué hace |
|---|---|---|
| **E1** | Eliminación de casos | Descarta las 13.396 filas incompletas. Es el conjunto que usaron S1 y S2. |
| **E2** | Imputación simple | Rellena con la mediana de `semestres_reales`. |
| **E3** | Imputación por regresión | Rellena con la predicción del modelo de la sección 3. |

In [ ]:
e1 = P.estrategia_eliminacion(df)
e2 = P.estrategia_simple(df, medida="mediana")
e3 = P.estrategia_regresion(df, ajuste)

print(f"E3 · casos imputados : {e3.attrs['n_imputados']:,}".replace(",", "."))
print(f"E3 · sin imputar     : {e3.attrs['n_sin_imputar']}")
assert e3.attrs["n_imputados"] == dx["n_faltantes"], "no se imputaron todos los faltantes"
print("\nTodos los faltantes quedaron imputados.")

In [ ]:
comparacion = P.comparar_estrategias(
    {"Eliminación": e1, "Imputación simple": e2, "Imputación por regresión": e3})
comparacion

In [ ]:
# Se guarda para los puntos 2 y 3, que reajustan el modelo sobre los tres conjuntos
comparacion.to_csv(RAIZ / "S3/resultados/comparacion_estrategias.csv", index=False)
print("escrito: S3/resultados/comparacion_estrategias.csv")

In [ ]:
ref = comparacion.iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

for a, (col, titulo) in zip(ax, [("media", "Media del atraso (semestres)"),
                                 ("% atraso alto", "% con atraso alto (> 4 semestres)")]):
    v = comparacion[col]
    colores = ["#777777", "#b22222", "#d95f02"]
    a.bar(range(3), v, color=colores)
    a.axhline(ref[col], color="#333333", ls="--", lw=1)
    a.text(2.45, ref[col], " S1/S2", va="center", fontsize=9)
    a.set_xticks(range(3))
    a.set_xticklabels(["Eliminación", "Imputación\nsimple", "Imputación\npor regresión"],
                      fontsize=9)
    a.set_title(titulo, fontsize=11)
    for i, x in enumerate(v):
        a.text(i, x, f"{x}", ha="center", va="bottom", fontsize=10)

fig.suptitle("La imputación simple desplaza la distribución; la regresión la conserva",
             fontsize=13)
plt.tight_layout()
plt.savefig(RAIZ / "S3/informe/figuras/s3_03_estrategias.png", dpi=150)
plt.show()

### Qué dice la comparación

**La imputación simple deforma los datos.** Sube la media del atraso de 3,40 a
3,79 e infla la proporción de atraso alto de 24,5 % a 32,7 % — ocho puntos
porcentuales que no existen. La razón es mecánica: asigna la misma duración
mediana a todos los casos incompletos, y como esos casos se concentran en
carreras de duración formal corta (a distancia y vespertina), la resta
`semestres_reales − dur_total_carr` produce atrasos artificialmente altos.

**Y destruye una correlación que la S2 había validado.** La relación entre
duración formal y atraso pasa de 0,009 —el valor que la S2 declaró «estable y
de magnitud despreciable», sosteniendo que *el atraso no depende del largo de
la carrera*— a **−0,218**. Es decir, el conjunto con imputación simple
respaldaría una conclusión que las dos entregas anteriores descartaron.

**La imputación por regresión conserva los hallazgos.** La media queda en 3,400
frente a 3,398 del conjunto original, el atraso alto en 23,8 % frente a 24,5 %,
y la correlación atraso–edad en 0,468 frente a 0,466. Recupera las 13.396
observaciones sin mover lo que S1 y S2 establecieron.

**Su límite, declarado.** La correlación entre semestres reales y edad baja de
0,461 a 0,175. Es el efecto esperable de sustituir valores observados por
predicciones, que tienen menos variabilidad que los datos reales; afecta sobre
todo a la modalidad a distancia, donde se imputan tres de cada cuatro casos.
Es una limitación real de E3 y se declara como tal.

## 5 · Qué conjunto se usa para el modelo logístico

**Se usa el conjunto con imputación por regresión (E3).** Tres razones, en
orden de peso:

1. **El mecanismo es MAR, no MCAR.** Eliminar los casos incompletos no quita
   un 12,75 % repartido al azar: quita el 75,9 % de la modalidad a distancia y
   el 1,5 % de la diurna. El conjunto E1 sobrerrepresenta a la jornada diurna,
   y como la jornada es una de las variables explicativas del modelo logístico,
   ese sesgo entraría directamente en los coeficientes.

2. **Preserva lo que S1 y S2 validaron**, mientras que la imputación simple
   invierte una de esas conclusiones. Un conjunto que contradice las entregas
   anteriores no puede ser la base de la tercera.

3. **Recupera 13.396 observaciones**, un 14,6 % más de datos, y justamente en
   las categorías donde E1 queda con menos respaldo muestral.

**Cómo se controla el riesgo.** El punto 3 de la pauta exige reajustar el
modelo logístico sobre los tres conjuntos y comparar coeficientes, amplitud de
intervalos, AUC y exactitud. Esa comparación es la que verifica si la elección
cambia las conclusiones; esta sección la deja preparada y no la anticipa.

## 6 · Partición entrenamiento / prueba

70 % y 30 % con `random_state=42`, como fija la pauta, **estratificada** por la
variable objetivo. La estratificación no es un detalle: `atraso_alto` ronda el
24 %, y una partición simple puede dejar proporciones distintas en cada parte,
lo que haría que las métricas de prueba no sean comparables con las de ajuste.

In [ ]:
train, test = P.particion(e3)
P.resumen_particion(train, test)

In [ ]:
train.to_parquet(RAIZ / "S3/resultados/train_e3.parquet", index=False)
test.to_parquet(RAIZ / "S3/resultados/test_e3.parquet", index=False)

for nombre, d in [("e1_eliminacion", e1), ("e2_simple", e2), ("e3_regresion", e3)]:
    d.to_parquet(RAIZ / f"S3/resultados/{nombre}.parquet", index=False)

print("Conjuntos guardados en S3/resultados/ para los puntos 2 y 3:")
for p in sorted((RAIZ / "S3/resultados").glob("*.parquet")):
    print(f"  {p.name}  ({p.stat().st_size // 1024} KB)")

## 7 · Resumen de la sección

| Tarea de la pauta | Resultado |
|---|---|
| Diagnóstico de la S1 resumido | 13.396 casos (12,75 %) con centinelas 1900 y 0 |
| Mecanismo de pérdida | **MAR**: la ausencia va de 1,5 % (diurna) a 75,9 % (a distancia) |
| Regresión lineal múltiple | 4 predictores, n = 91.667, **R² = 0,79**, **RMSE = 2,31**, MAE = 0,98 |
| Supuestos | Colinealidad correcta (VIF máx. 2,42); normalidad y homocedasticidad no se cumplen, con su efecto acotado |
| Casos imputados | **13.396**, el total de los faltantes |
| Comparación de 3 estrategias | Tabla con n, media, desviación, asimetría y tres correlaciones |
| Conjunto elegido | **E3, imputación por regresión**, con su limitación declarada |
| Partición | 70/30 estratificada, semilla 42, 23,84 % de atraso alto en ambas partes |

**Para quien sigue con el punto 2:** los tres conjuntos y la partición están en
`S3/resultados/` en formato parquet. La variable objetivo es `atraso_alto`
(Int64, 0/1) y el conjunto base es `e3_regresion.parquet`.